# Lab 08｜按需调阅关键上下文：一项 Skill 的内容怎样逐层出现？

这是专栏《自进化 Agent 实战》08 讲《渐进披露：按需调阅关键上下文》的配套练习。练习使用构造的事故手册材料，目标是掌握处理真实场景的能力。

## 题面

一套事故手册越攒越多：调查主干步骤、旧日志的字段对照表、报告模板、引用检查脚本都想交给 Agent。有人主张把它们全部写进一份方法文件，这样 Agent 一上手就什么都知道。可你注意到：即使这次告警还没走到写报告那一步，模板和检查说明也跟着占了每次调用的位置。你会怎样组织这些材料，让 Agent 在调查的不同阶段只拿到当时需要的那一部分？又怎样确认它是真的按进度去读资料、跑脚本，而不是一开始就被塞满？

## 参考答案

先把材料按"什么时候用"分层。主干步骤留在入口文件里；较长的字段说明、报告模板、可重复执行的检查脚本分别放在独立文件。这样做的目的不是整理目录，而是控制每次调用真正带进去的内容。

任务刚开始时，不必交出每项能力的完整正文，只要先给出名称、一句说明它处理什么、什么时候适用，让模型知道有哪些能力可用就够了。等告警和这句说明对上，模型选定这项能力，完整方法才进入接下来的输入。

进入方法以后，也不是一次性把所有附件都倒进去。模型走到读旧日志这一步，才去读字段对照表；要形成报告时才打开模板；报告写完，才提出运行检查脚本。这几次取用都是模型在循环里主动发起的：它提出读文件或跑命令，运行环境才执行，再把结果带回下一次调用。所以什么时候取用，跟着调查进度走，而不是运行环境预先猜该塞什么。

验证时我会看运行记录的先后：最初只有名称和说明；完整方法从选中之后才开始影响行动；参考资料等到对应步骤才被读取；脚本的源码不常驻，回到输入的只是执行结果。这样字段口径变了只改对照表，脚本变了只换脚本，主干方法不用重写。

## 准备

使用 Python 3.12 内核；实际模型运行需要 `GLM_API_KEY`，未设置时弹出隐藏输入框，依赖自动补装。`lab.codex` 是真实 SDK 客户端；下面分别新建 Thread，只提交普通任务文本，让模型根据能力描述选择。附件由模型在工具循环中读取，Notebook 的本地展示不会自动成为模型输入。
`lab.watch()` 只做超时保护，`lab.record_result()` 只保存结果。离线验证不发起模型请求；最后一格关闭连接。


In [ ]:
import sys
from pathlib import Path
examples = next(p for base in (Path.cwd(), *Path.cwd().parents)
                for p in (base, base / "examples") if (p / "notebook08_setup.py").is_file())
sys.path.insert(0, str(examples))
from notebook08_setup import prepare
story = prepare()

## 阶段一｜一项 Skill 变长以后，为什么不能一次全部装入 Context？

读取同一套构造材料，比较主干入口与“字段表、模板、源码全堆进入口”的字符数。
跑完看提前装入了多少附件：Context 预算限制的是每次真正携带的内容，字符数只用于本地对照。
这里起作用的是 Context 预算与分层；目录拆开后，仍要由 Loop 决定取用时机。

In [ ]:
from IPython.display import Markdown, display
import subprocess
skill_dir, skill_path = story.skill_dir, story.skill_path
names = ("SKILL.md", "references/log-fields.md", "assets/incident-report.md", "scripts/check_report.py")
contents = {name: (skill_dir / name).read_text(encoding="utf-8") for name in names}
core = contents["SKILL.md"]
packed = "\n\n".join(contents.values())
display(Markdown("### 主干入口\n```text\n" + core + "\n```"))

In [ ]:
extra_chars = len(packed) - len(core)
print("主干入口字符数：", len(core), "；全部堆入字符数：", len(packed))
print("尚未需要、却提前携带的附件字符数：", extra_chars)
story.show_budget(contents, packed, core)

## 阶段二｜渐进披露怎样借助 Agent Loop 发生？

在两个互不继承历史的 Thread 中，先提交无关任务，再提交与描述匹配的调查任务。不指定 Skill 名称或路径，也不使用 SkillInput 强选。
跑完对照各次输入中的正文标记：路径不算正文，脚本结果与源码分开看；litellm 代理层记录实际模型请求。
分别保留是否读取正文、读取了哪些附件的实际轨迹。未触发或误触发都如实显示；离线检查不生成选择结果。

In [ ]:
from openai_codex import ApprovalMode, Codex, Sandbox, TextInput
lab, work = story.lab, story.work
codex: Codex = lab.codex if lab else None
if not lab:
    raise RuntimeError("离线检查不生成模型选择结果；请运行 run_validation.py --mode offline。")
unrelated = codex.thread_start(cwd=str(work), sandbox=Sandbox.full_access, approval_mode=ApprovalMode.deny_all)
unrelated_start = len(lab.requests)
turn = unrelated.turn(TextInput("给三张旅行照片起简短中文标题：海边日出、雨后街道、山顶云海。只给建议，不修改文件。"))
with lab.watch(turn):
    unrelated_result = turn.run()
lab.record_result(turn, unrelated_result, show_result=False)
print(lab.clean(unrelated_result.final_response))
story.ranges["unrelated"] = [unrelated_start, len(lab.requests)]
story.save("request-ranges.json", story.ranges)


In [ ]:
thread = codex.thread_start(cwd=str(work), sandbox=Sandbox.full_access, approval_mode=ApprovalMode.deny_all)
loop_start = len(lab.requests)
turn = thread.turn(TextInput("调查历史支付模块当前超时告警，起点是 inputs/alert.md。"
    "核对当前发布、旧日志、连接池与运行配置，区分事实、推断和待验证问题，写入 report.md 并检查引用。"
    "只调查本次构造材料，不修改输入、方法或配置，不读项目外文件，不访问网络。"))
with lab.watch(turn):
    result = turn.run()
lab.record_result(turn, result, show_result=False)
story.ranges["matching"] = [loop_start, len(lab.requests)]
story.save("selection-threads.json", {"unrelated": unrelated.id, "matching": thread.id, "explicit_skill_input": False})
story.observe_loop(loop_start)


## 阶段三｜Harness 怎样把 Skill 接入这条 Loop？

按两种任务各自的实际请求范围，检查是否看见能力索引、是否读取了方法正文，以及附件和脚本何时返回。
跑完把能力索引、选中装入、Loop 按需取用和 litellm 代理层的职责对回前面的输入记录。
这里起作用的是发现机制与入口描述；参考和脚本不会随目录自动装入，描述边界要另行检查。

In [ ]:
import json
from notebook_support import command_receipts, tool_output_contains
observations = {}
for name in ("unrelated", "matching"):
    requests = lab.requests[slice(*story.ranges[name])]
    receipts = command_receipts(requests)
    observations[name] = {"requests": len(requests),
        "index_seen": any(story.markers["META"] in json.dumps(r, ensure_ascii=False) for r in requests),
        "body_read": any(r["exit_code"] == 0 and "SKILL.md" in r["command"]
                         and story.markers["BODY"] in r["output"] for r in receipts),
        "first_tool_result": {key: next((i for i, r in enumerate(requests, 1)
            if tool_output_contains(r, marker)), None) for key, marker in story.markers.items()},
        "receipts": receipts}
story.save("selection-observations.json", observations)
story.table(["任务", "索引可见", "正文有成功读取回执", "各层首次工具回执"],
    [(name, v["index_seen"], v["body_read"], v["first_tool_result"]) for name, v in observations.items()],
    "这是本轮真实轨迹，不是词语匹配预测。未见回执不等于已选用；正文可见与成功读取也要分开。")

In [ ]:
from notebook08_story import snapshot
try:
    after = {"inputs": snapshot(work / "inputs"), "skill": snapshot(skill_dir)}
    story.save("inputs-after.json", after)
    print("输入与方法未改变：", after == story.initial_files)
    print("报告已写入：", story.report_path.is_file() and bool(story.report_path.read_text().strip()))
    print("本次模型请求数：", len(lab.requests))
    story.phase("阶段三")
finally:
    story.close()